# Notebook 10: SRS-Adaptive MAML Ablation — XuetangX

**Purpose:** Ablation — tests SRS-Adaptive inner loop **WITHOUT** warm-start. Implements Contribution C2.

**Key differences from NB07 (Standard MAML):**
- Inner loop: SRS-Adaptive (αᵢ = α_base × SRSᵢ, Kᵢ = f(SRSᵢ, τ))
- Outer loop: SRS-weighted gradient sum (θ ← θ − β · ∇_θ Σᵢ SRSᵢ · L_query(φᵢ))
- NO warm-start (random init, same as NB07)
- OUTER_LR = 0.001 (same as NB07 — no warm-start here)

**Inputs:**
- `data/processed/xuetangx/vocab/course2id.json`
- `data/processed/xuetangx/episodes/episodes_{train|val|test}_K5_Q10.parquet`
- `data/processed/xuetangx/pairs_srs/pairs.parquet`

**Outputs:**
- `models/maml/maml_srs_adaptive_xuetangx.pth` (best checkpoint)
- `reports/10_srs_adaptive_maml_xuetangx/<run_tag>/results_10_srs_adaptive_maml_xuetangx.json`

**Next:** `11_warmstart_srs_adaptive_maml_xuetangx.ipynb` — Full combined contribution (C3)

In [1]:
# [CELL 10-00] Bootstrap: repo root + paths + helpers

import os
# Required for deterministic CuBLAS on CUDA >= 10.2
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
import json
import time
import uuid
import math
import copy
import pickle
import hashlib
import random
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Tuple, Optional
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"[CELL 10-00] start={datetime.now().isoformat(timespec='seconds')}")
print("[CELL 10-00] CWD:", Path.cwd().resolve())


def find_repo_root(start: Path) -> Path:
    """Search upward for PROJECT_STATE.md — single source of truth."""
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / "PROJECT_STATE.md").exists():
            return p
    raise RuntimeError(
        "Could not find PROJECT_STATE.md in current or parent directories. "
        "Open this notebook from inside the repo."
    )


REPO_ROOT = find_repo_root(Path.cwd())

PATHS = {
    "PROJECT_STATE":  REPO_ROOT / "PROJECT_STATE.md",
    "META_REGISTRY":  REPO_ROOT / "meta.json",
    "DATA_RAW":       REPO_ROOT / "data" / "raw",
    "DATA_INTERIM":   REPO_ROOT / "data" / "interim",
    "DATA_PROCESSED": REPO_ROOT / "data" / "processed",
    "NOTEBOOKS":      REPO_ROOT / "notebooks",
    "REPORTS":        REPO_ROOT / "reports",
    "MODELS":         REPO_ROOT / "models",
    "RUNS":           REPO_ROOT / "runs",
}

for p in PATHS.values():
    Path(p).mkdir(parents=True, exist_ok=True) if str(p).endswith(("reports", "models", "runs")) else None
(PATHS["MODELS"] / "baselines").mkdir(parents=True, exist_ok=True)
(PATHS["MODELS"] / "maml").mkdir(parents=True, exist_ok=True)
(PATHS["REPORTS"]).mkdir(parents=True, exist_ok=True)


def cell_start(cid: str, title: str, **kw) -> float:
    t = time.time()
    print(f"\n[{cid}] {title}")
    print(f"[{cid}] start={datetime.now().isoformat(timespec='seconds')}")
    for k, v in kw.items():
        print(f"[{cid}] {k}={v}")
    return t


def cell_end(cid: str, t0: float, **kw) -> None:
    for k, v in kw.items():
        print(f"[{cid}] {k}={v}")
    print(f"[{cid}] elapsed={time.time()-t0:.2f}s  done")


def write_json_atomic(path: Path, obj: Any, indent: int = 2) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp_{uuid.uuid4().hex}")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding="utf-8")
    tmp.replace(path)


def read_json(path: Path) -> Any:
    path = Path(path)
    if not path.exists():
        raise RuntimeError(f"Missing required JSON file: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


print(f"[CELL 10-00] REPO_ROOT={REPO_ROOT}")
print("[CELL 10-00] done")

[CELL 10-00] start=2026-04-09T15:57:05
[CELL 10-00] CWD: /home/user/jamalla/anonymous-users-mooc-session-meta/notebooks/xuetangx
[CELL 10-00] REPO_ROOT=/home/user/jamalla/anonymous-users-mooc-session-meta
[CELL 10-00] done


In [2]:
# [CELL 10-01] Seed + global constants

t0 = cell_start("CELL 10-01", "Seed everything + global constants")

GLOBAL_SEED   = 20260107
NOTEBOOK_NAME = "10_srs_adaptive_maml_xuetangx"
DATASET       = "xuetangx"


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print(f"[CELL 10-01] WARN: use_deterministic_algorithms: {e}")


seed_everything(GLOBAL_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cell_end("CELL 10-01", t0, seed=GLOBAL_SEED, device=DEVICE)


[CELL 10-01] Seed everything + global constants
[CELL 10-01] start=2026-04-09T15:57:05
[CELL 10-01] seed=20260107
[CELL 10-01] device=cuda
[CELL 10-01] elapsed=0.10s  done


In [3]:
# [CELL 10-02] Run config + data paths
#
# KEY DIFFERENCES FROM NB07 (Standard MAML):
#   - SRS-Adaptive inner loop: alpha_i = ALPHA_BASE * srs_i
#   - Task-specific K: K_MIN if srs_i >= TAU else K_MAX
#   - SRS-weighted outer loss: total_loss += srs_i * query_loss
#   - OUTER_LR = 0.001  (SAME as NB07 -- no warm-start, random init)
#   - NO pretrained weights loaded

t0 = cell_start("CELL 10-02", "Run config + paths")

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ID  = uuid.uuid4().hex

OUT_DIR       = PATHS["REPORTS"] / NOTEBOOK_NAME / RUN_TAG
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH   = OUT_DIR / "report.json"
CONFIG_PATH   = OUT_DIR / "config.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

# -- Data paths ------------------------------------------------
EPISODES_DIR = PATHS["DATA_PROCESSED"] / "xuetangx" / "episodes"
PAIRS_SRS_DIR = PATHS["DATA_PROCESSED"] / "xuetangx" / "pairs_srs"
VOCAB_DIR    = PATHS["DATA_PROCESSED"] / "xuetangx" / "vocab"
MODELS_DIR   = PATHS["MODELS"] / "maml"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

K_SUPPORT = 5
Q_QUERY   = 10

EPISODES_TRAIN  = EPISODES_DIR / f"episodes_train_K{K_SUPPORT}_Q{Q_QUERY}.parquet"
EPISODES_VAL    = EPISODES_DIR / f"episodes_val_K{K_SUPPORT}_Q{Q_QUERY}.parquet"
EPISODES_TEST   = EPISODES_DIR / f"episodes_test_K{K_SUPPORT}_Q{Q_QUERY}.parquet"
PAIRS_SRS_PATH  = PAIRS_SRS_DIR / "pairs.parquet"
VOCAB_PATH      = VOCAB_DIR / "course2id.json"
CHECKPOINT_PATH = MODELS_DIR / f"maml_srs_adaptive_{DATASET}.pth"
RESULTS_PATH    = OUT_DIR / f"results_{NOTEBOOK_NAME}.json"

# -- Hyperparameters (CLAUDE.md locked) -----------------------
ALPHA_BASE  = 0.01    # base inner-loop step size
TAU         = 0.5     # SRS threshold for K selection
K_MIN       = 3       # inner steps for high-SRS sessions (SRS >= TAU)
K_MAX       = 7       # inner steps for low-SRS  sessions (SRS < TAU)
OUTER_LR    = 0.001   # same as NB07 -- no warm-start
BATCH_SIZE  = 32
N_ITER      = 3000
VAL_EVERY   = 100
MAX_SEQ_LEN = 50

# -- GRU architecture (locked -- same as NB06/NB07/NB08) ------
GRU_CFG = {
    "embedding_dim": 64,
    "hidden_dim":    128,
    "num_layers":    1,
    "dropout":       0.0,
}

CFG = {
    "notebook":        NOTEBOOK_NAME,
    "dataset":         DATASET,
    "global_seed":     GLOBAL_SEED,
    "device":          DEVICE,
    "K_support":       K_SUPPORT,
    "Q_query":         Q_QUERY,
    "alpha_base":      ALPHA_BASE,
    "tau":             TAU,
    "k_min":           K_MIN,
    "k_max":           K_MAX,
    "outer_lr":        OUTER_LR,
    "batch_size":      BATCH_SIZE,
    "n_iterations":    N_ITER,
    "val_every":       VAL_EVERY,
    "max_seq_len":     MAX_SEQ_LEN,
    "gru_cfg":         GRU_CFG,
    "warmstart":       False,
    "srs_adaptive":    True,
    "inputs": {
        "episodes_train": str(EPISODES_TRAIN),
        "episodes_val":   str(EPISODES_VAL),
        "episodes_test":  str(EPISODES_TEST),
        "pairs_srs":      str(PAIRS_SRS_PATH),
        "vocab":          str(VOCAB_PATH),
    },
}

write_json_atomic(CONFIG_PATH, CFG)

# -- Init report / manifest -----------------------------------
write_json_atomic(REPORT_PATH, {
    "run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "run_tag": RUN_TAG,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "metrics": {}, "key_findings": [], "sanity_samples": {},
    "data_fingerprints": {}, "notes": [],
})
write_json_atomic(MANIFEST_PATH, {
    "run_id": RUN_ID, "notebook": NOTEBOOK_NAME, "run_tag": RUN_TAG, "artifacts": []
})

# -- Append to meta.json --------------------------------------
META = PATHS["META_REGISTRY"]
if not META.exists():
    write_json_atomic(META, {"schema_version": 1, "runs": []})
m = read_json(META)
m["runs"].append({"run_id": RUN_ID, "notebook": NOTEBOOK_NAME,
                   "run_tag": RUN_TAG, "out_dir": str(OUT_DIR)})
write_json_atomic(META, m)

print(f"[CELL 10-02] ABLATION: SRS-Adaptive MAML (NO warm-start)")
print(f"[CELL 10-02] alpha_base={ALPHA_BASE}  tau={TAU}  K_min={K_MIN}  K_max={K_MAX}")
print(f"[CELL 10-02] outer_lr={OUTER_LR}  batch_size={BATCH_SIZE}  n_iterations={N_ITER}  val_every={VAL_EVERY}")
print(f"[CELL 10-02] warmstart=False  srs_adaptive=True")
cell_end("CELL 10-02", t0, out_dir=str(OUT_DIR))


[CELL 10-02] Run config + paths
[CELL 10-02] start=2026-04-09T15:57:05
[CELL 10-02] ABLATION: SRS-Adaptive MAML (NO warm-start)
[CELL 10-02] alpha_base=0.01  tau=0.5  K_min=3  K_max=7
[CELL 10-02] outer_lr=0.001  batch_size=32  n_iterations=3000  val_every=100
[CELL 10-02] warmstart=False  srs_adaptive=True
[CELL 10-02] out_dir=/home/user/jamalla/anonymous-users-mooc-session-meta/reports/10_srs_adaptive_maml_xuetangx/20260409_155705
[CELL 10-02] elapsed=0.02s  done


In [4]:
# [CELL 10-03] Load data: vocab, episodes, pairs_srs

t0 = cell_start("CELL 10-03", "Load vocab + episodes + pairs_srs")

# -- Guard: required files must exist -------------------------
for label, path in [
    ("vocab",          VOCAB_PATH),
    ("episodes_train", EPISODES_TRAIN),
    ("episodes_val",   EPISODES_VAL),
    ("episodes_test",  EPISODES_TEST),
    ("pairs_srs",      PAIRS_SRS_PATH),
]:
    if not Path(path).exists():
        raise RuntimeError(
            f"Missing {label}: {path}\n"
            "Run notebooks 01->02->03->03b->04->05 first."
        )

# -- Vocab ----------------------------------------------------
course2id = read_json(VOCAB_PATH)
id2course  = {int(v): k for k, v in course2id.items()}
n_items    = len(course2id)
print(f"[CELL 10-03] Vocabulary: {n_items} courses")

# -- Episodes -------------------------------------------------
episodes_train = pd.read_parquet(EPISODES_TRAIN)
episodes_val   = pd.read_parquet(EPISODES_VAL)
episodes_test  = pd.read_parquet(EPISODES_TEST)

print(f"[CELL 10-03] episodes_train: {len(episodes_train):,} ({episodes_train['user_id'].nunique():,} users)")
print(f"[CELL 10-03] episodes_val:   {len(episodes_val):,} ({episodes_val['user_id'].nunique():,} users)")
print(f"[CELL 10-03] episodes_test:  {len(episodes_test):,} ({episodes_test['user_id'].nunique():,} users)")

# -- Pairs with SRS -------------------------------------------
pairs_srs = pd.read_parquet(PAIRS_SRS_PATH)
print(f"[CELL 10-03] pairs_srs: {len(pairs_srs):,} rows, columns: {list(pairs_srs.columns)}")

# Validate SRS column exists
assert "srs" in pairs_srs.columns, "'srs' column missing from pairs_srs. Run 03b_srs_scores first."
assert "pair_id" in pairs_srs.columns, "'pair_id' column missing from pairs_srs."
assert "prefix" in pairs_srs.columns, "'prefix' column missing from pairs_srs."
assert "label" in pairs_srs.columns, "'label' column missing from pairs_srs."

print(f"[CELL 10-03] SRS in pairs_srs: mean={pairs_srs['srs'].mean():.4f}  "
      f"p50={pairs_srs['srs'].quantile(0.5):.4f}  "
      f"min={pairs_srs['srs'].min():.4f}  max={pairs_srs['srs'].max():.4f}")

# -- Index pairs_srs by pair_id for fast lookup ---------------
pairs_srs_idx = pairs_srs.set_index("pair_id")
print(f"[CELL 10-03] pairs_srs indexed by pair_id.")

cell_end("CELL 10-03", t0, n_items=n_items)


[CELL 10-03] Load vocab + episodes + pairs_srs
[CELL 10-03] start=2026-04-09T15:57:05
[CELL 10-03] Vocabulary: 1517 courses
[CELL 10-03] episodes_train: 113,920 (4,645 users)
[CELL 10-03] episodes_val:   942 (942 users)
[CELL 10-03] episodes_test:  986 (986 users)
[CELL 10-03] pairs_srs: 487,696 rows, columns: ['pair_id', 'user_id', 'session_id', 'prefix', 'label', 'label_ts_epoch', 'prefix_len', 'srs', 'srs_intensity', 'srs_extent', 'srs_composition']
[CELL 10-03] SRS in pairs_srs: mean=0.4887  p50=0.4284  min=0.0614  max=1.0000
[CELL 10-03] pairs_srs indexed by pair_id.
[CELL 10-03] n_items=1517
[CELL 10-03] elapsed=0.57s  done


In [5]:
# [CELL 10-04] Attach SRS to episodes
#
# Each episode's SRS = mean SRS of its K support pairs.
# This is the task-level SRS used in get_task_hyperparams() and the outer loop.
# Join is on pair_id from pairs_srs_idx.

t0 = cell_start("CELL 10-04", "Attach SRS to episodes (mean SRS of support pairs)")

import ast


def get_episode_srs(support_pair_ids, pairs_idx: pd.DataFrame) -> float:
    """
    Compute the mean SRS of the support pairs for one episode.
    Falls back to 0.5 (neutral) for any missing pair_ids.
    """
    if isinstance(support_pair_ids, str):
        support_pair_ids = ast.literal_eval(support_pair_ids)
    srs_vals = []
    for pid in support_pair_ids:
        if pid in pairs_idx.index:
            srs_vals.append(float(pairs_idx.loc[pid, "srs"]))
        else:
            srs_vals.append(0.5)  # neutral fallback
    return float(np.mean(srs_vals)) if srs_vals else 0.5


# Attach SRS to all three splits
episodes_train = episodes_train.copy()
episodes_val   = episodes_val.copy()
episodes_test  = episodes_test.copy()

episodes_train["srs"] = episodes_train["support_pair_ids"].apply(
    lambda ids: get_episode_srs(ids, pairs_srs_idx)
)
episodes_val["srs"] = episodes_val["support_pair_ids"].apply(
    lambda ids: get_episode_srs(ids, pairs_srs_idx)
)
episodes_test["srs"] = episodes_test["support_pair_ids"].apply(
    lambda ids: get_episode_srs(ids, pairs_srs_idx)
)

print(f"[CELL 10-04] Train episode SRS: mean={episodes_train['srs'].mean():.4f}  "
      f"p50={episodes_train['srs'].median():.4f}  "
      f"min={episodes_train['srs'].min():.4f}  max={episodes_train['srs'].max():.4f}")
print(f"[CELL 10-04] Val episode SRS:   mean={episodes_val['srs'].mean():.4f}")
print(f"[CELL 10-04] Test episode SRS:  mean={episodes_test['srs'].mean():.4f}")

# Tier counts (train)
n_low    = (episodes_train["srs"] < TAU).sum()
n_high   = (episodes_train["srs"] >= TAU).sum()
print(f"[CELL 10-04] Train episodes: low-SRS (K={K_MAX} steps): {n_low:,}  "
      f"high-SRS (K={K_MIN} steps): {n_high:,}")

cell_end("CELL 10-04", t0)


[CELL 10-04] Attach SRS to episodes (mean SRS of support pairs)
[CELL 10-04] start=2026-04-09T15:57:06
[CELL 10-04] Train episode SRS: mean=0.6157  p50=0.6153  min=0.0614  max=1.0000
[CELL 10-04] Val episode SRS:   mean=0.5353
[CELL 10-04] Test episode SRS:  mean=0.5355
[CELL 10-04] Train episodes: low-SRS (K=7 steps): 43,036  high-SRS (K=3 steps): 70,884
[CELL 10-04] elapsed=4.97s  done


In [6]:
# [CELL 10-05] GRURecommender model definition
#
# Identical architecture to NB06/NB07/NB08 (locked).
# No pretrained weights loaded here (unlike NB08/NB11).

t0 = cell_start("CELL 10-05", "Define GRURecommender")


class GRURecommender(nn.Module):
    """GRU-based sequential recommender.

    Architecture (matches NB06/NB07 -- LOCKED):
        Embedding(n_items, embedding_dim) -> GRU(embedding_dim, hidden_dim) -> Linear(hidden_dim, n_items)
    """

    def __init__(
        self,
        n_items: int,
        embedding_dim: int = 64,
        hidden_dim: int = 128,
        num_layers: int = 1,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.n_items       = n_items
        self.embedding_dim = embedding_dim
        self.hidden_dim    = hidden_dim
        self.num_layers    = num_layers

        self.embedding = nn.Embedding(n_items + 1, embedding_dim, padding_idx=0)
        gru_dropout = dropout if num_layers > 1 else 0.0
        self.gru = nn.GRU(
            embedding_dim, hidden_dim, num_layers,
            batch_first=True, dropout=gru_dropout
        )
        self.fc = nn.Linear(hidden_dim, n_items)

    def forward(self, seq: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        """
        Args:
            seq:     (batch, max_len) padded item-id sequences
            lengths: (batch,) actual sequence lengths
        Returns:
            logits: (batch, n_items)
        """
        emb = self.embedding(seq)                        # (B, L, E)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, h_n = self.gru(packed)                        # h_n: (layers, B, H)
        h_last = h_n[-1]                                 # (B, H)
        return self.fc(h_last)                           # (B, n_items)


# Quick sanity check
_m = GRURecommender(n_items=100, **GRU_CFG)
_x = torch.randint(1, 100, (4, 10))
_l = torch.tensor([10, 8, 5, 3])
_o = _m(_x, _l)
assert _o.shape == (4, 100), f"Expected (4,100) got {_o.shape}"
del _m, _x, _l, _o

cell_end("CELL 10-05", t0)


[CELL 10-05] Define GRURecommender
[CELL 10-05] start=2026-04-09T15:57:11
[CELL 10-05] elapsed=0.05s  done


In [7]:
# [CELL 10-06] functional_forward — uses torch.func.functional_call (fast CUDA GRU)

t0 = cell_start("CELL 10-06", "Define functional_forward for FOMAML")

from torch.func import functional_call

def functional_forward(
    model: GRURecommender,
    params: Dict[str, torch.Tensor],
    seqs: torch.Tensor,
    lengths: torch.Tensor,
) -> torch.Tensor:
    """
    Functional forward using torch.func.functional_call.
    Runs the model native optimized CUDA GRU with the given params dict.
    Vastly faster than manual step-by-step GRU loop.
    """
    return functional_call(model, params, (seqs, lengths))


cell_end("CELL 10-06", t0)



[CELL 10-06] Define functional_forward for FOMAML
[CELL 10-06] start=2026-04-09T15:57:11
[CELL 10-06] elapsed=0.00s  done


In [8]:
# [CELL 10-07] SRS-Adaptive functions
#
# get_task_hyperparams: compute alpha_i and K_i from SRS score
# srs_adaptive_inner_loop: MAML inner loop with task-specific alpha and K
# outer_step: SRS-weighted outer gradient update
#
# All three functions match CLAUDE.md specification exactly.

t0 = cell_start("CELL 10-07", "Define SRS-Adaptive functions")


def get_task_hyperparams(
    srs_i: float,
    alpha_base: float = ALPHA_BASE,
    tau: float = TAU,
    k_min: int = K_MIN,
    k_max: int = K_MAX,
) -> Tuple[float, int]:
    """
    Compute task-specific inner-loop hyperparameters from SRS score.

    alpha_i = alpha_base * SRS_i
    K_i     = K_min if SRS_i >= tau else K_max

    High SRS -> larger alpha, fewer steps (reliable session, fast adaptation)
    Low SRS  -> smaller alpha, more steps  (noisy session, careful adaptation)

    Args:
        srs_i:      float in [0, 1], SRS score for this task
        alpha_base: base inner-loop step size (ALPHA_BASE = 0.01)
        tau:        SRS threshold for K selection (TAU = 0.5)
        k_min:      inner steps for high-SRS tasks (K_MIN = 3)
        k_max:      inner steps for low-SRS  tasks (K_MAX = 7)

    Returns:
        (alpha_i, K_i)
    """
    alpha_i = alpha_base * srs_i
    K_i     = k_min if srs_i >= tau else k_max
    return float(alpha_i), int(K_i)


def srs_adaptive_inner_loop(
    model: GRURecommender,
    params: Dict[str, torch.Tensor],
    support_seqs: torch.Tensor,
    support_lengths: torch.Tensor,
    support_labels: torch.Tensor,
    alpha_i: float,
    K_i: int,
) -> Dict[str, torch.Tensor]:
    """
    Standard MAML inner loop with task-specific alpha and K.
    alpha_i and K_i are computed from the session SRS score.
    Updates params and returns adapted phi_i.

    Args:
        model:           GRURecommender (architecture reference only)
        params:          dict of current meta-parameters (cloned)
        support_seqs:    (K, max_len) support sequence tensor
        support_lengths: (K,) support sequence lengths
        support_labels:  (K,) support label ids
        alpha_i:         task-specific inner-loop step size
        K_i:             task-specific number of inner gradient steps

    Returns:
        phi_i: adapted parameter dict after K_i gradient steps
    """
    for step in range(K_i):
        logits = functional_forward(model, params, support_seqs, support_lengths)
        loss   = F.cross_entropy(logits, support_labels)
        grads  = torch.autograd.grad(loss, params.values(), create_graph=False)
        params = {n: p - alpha_i * g
                  for (n, p), g in zip(params.items(), grads)}
    return params


def outer_step(
    meta_model: GRURecommender,
    batch_tasks: List[Dict],
    meta_optimizer: torch.optim.Optimizer,
) -> float:
    """
    Perform one outer meta-gradient update with SRS-weighted loss.

    theta <- theta - beta * grad_theta sum_i SRS_i * L_query(phi_i)
    SRS-weighted sum: high-reliability tasks have stronger influence on theta.

    Args:
        meta_model:     GRURecommender (meta-parameters theta)
        batch_tasks:    list of task dicts, each with keys:
                            srs, support_seqs, support_lengths, support_labels,
                            query_seqs, query_lengths, query_labels
        meta_optimizer: outer optimizer (Adam, lr=OUTER_LR)

    Returns:
        total_loss (float, unnormalized SRS-weighted sum)
    """
    meta_optimizer.zero_grad()
    total_loss = torch.tensor(0.0, requires_grad=True)

    for task in batch_tasks:
        srs_i        = task["srs"]                     # float in [0, 1]
        alpha_i, K_i = get_task_hyperparams(srs_i)
        params       = {n: p.clone() for n, p in meta_model.named_parameters()}
        phi_i        = srs_adaptive_inner_loop(
                           meta_model, params,
                           task["support_seqs"],
                           task["support_lengths"],
                           task["support_labels"],
                           alpha_i, K_i)
        query_logits = functional_forward(meta_model, phi_i,
                           task["query_seqs"],
                           task["query_lengths"])
        query_loss   = F.cross_entropy(query_logits, task["query_labels"])
        total_loss   = total_loss + srs_i * query_loss  # SRS-weighted

    total_loss.backward()
    meta_optimizer.step()
    return total_loss.item()


# -- Quick sanity check: get_task_hyperparams -----------------
a_hi, k_hi = get_task_hyperparams(0.8)   # high SRS
a_lo, k_lo = get_task_hyperparams(0.2)   # low SRS
assert k_hi == K_MIN, f"Expected K_MIN={K_MIN} for high SRS, got {k_hi}"
assert k_lo == K_MAX, f"Expected K_MAX={K_MAX} for low SRS, got {k_lo}"
assert abs(a_hi - ALPHA_BASE * 0.8) < 1e-9, "alpha_i mismatch for high SRS"
assert abs(a_lo - ALPHA_BASE * 0.2) < 1e-9, "alpha_i mismatch for low SRS"
print(f"[CELL 10-07] Sanity: high-SRS(0.8) -> alpha={a_hi:.4f}, K={k_hi}  (fast adapt)")
print(f"[CELL 10-07] Sanity: low-SRS(0.2)  -> alpha={a_lo:.4f}, K={k_lo}  (careful adapt)")

cell_end("CELL 10-07", t0)


[CELL 10-07] Define SRS-Adaptive functions
[CELL 10-07] start=2026-04-09T15:57:11
[CELL 10-07] Sanity: high-SRS(0.8) -> alpha=0.0080, K=3  (fast adapt)
[CELL 10-07] Sanity: low-SRS(0.2)  -> alpha=0.0020, K=7  (careful adapt)
[CELL 10-07] elapsed=0.00s  done


In [9]:
# [CELL 10-08] Evaluation metrics: HR@1/5/10, NDCG@10, MRR

t0 = cell_start("CELL 10-08", "Define evaluation metrics")


def ndcg_at_k(ranked_items: np.ndarray, true_item: int, k: int = 10) -> float:
    """NDCG@K = 1/log2(rank+2) if true_item in top-K, else 0."""
    for i, item in enumerate(ranked_items[:k]):
        if item == true_item:
            return 1.0 / math.log2(i + 2)
    return 0.0


def compute_metrics(
    scores: np.ndarray,
    labels: np.ndarray,
) -> Dict[str, float]:
    """
    Compute HR@1, HR@5, HR@10, NDCG@10, MRR.

    Args:
        scores: (n_samples, n_items) float score matrix (higher = more relevant)
        labels: (n_samples,) int true item indices

    Returns:
        dict with keys: hr1, hr5, hr10, ndcg10, mrr, n
    """
    n = len(labels)
    ranked = np.argsort(-scores, axis=1)   # descending rank

    hr1_vals    = []
    hr5_vals    = []
    hr10_vals   = []
    ndcg10_vals = []
    mrr_vals    = []

    for i in range(n):
        true     = int(labels[i])
        rank_arr = ranked[i]

        hr1_vals.append(1.0 if rank_arr[0] == true else 0.0)
        hr5_vals.append(1.0 if true in rank_arr[:5] else 0.0)
        hr10_vals.append(1.0 if true in rank_arr[:10] else 0.0)
        ndcg10_vals.append(ndcg_at_k(rank_arr, true, k=10))

        # MRR: reciprocal rank (unbounded)
        pos = np.where(rank_arr == true)[0]
        mrr_vals.append(1.0 / (pos[0] + 1) if len(pos) > 0 else 0.0)

    return {
        "hr1":    float(np.mean(hr1_vals)),
        "hr5":    float(np.mean(hr5_vals)),
        "hr10":   float(np.mean(hr10_vals)),
        "ndcg10": float(np.mean(ndcg10_vals)),
        "mrr":    float(np.mean(mrr_vals)),
        "n":      n,
    }


cell_end("CELL 10-08", t0)


[CELL 10-08] Define evaluation metrics
[CELL 10-08] start=2026-04-09T15:57:11
[CELL 10-08] elapsed=0.00s  done


In [10]:
# [CELL 10-09] Episode collation helpers
#
# pad_sequences: pad variable-length sequences to a fixed max_len
# get_episode_data: fetch support + query tensors for one episode,
#                   including the episode's SRS value for the inner loop.

t0 = cell_start("CELL 10-09", "Define pad_sequences + get_episode_data")


def pad_sequences(
    sequences: List[List[int]],
    max_len: int,
    pad_value: int = 0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Pad a list of variable-length sequences to max_len.

    Returns:
        padded:  (batch, max_len) LongTensor
        lengths: (batch,)         LongTensor of actual lengths (clipped to max_len)
    """
    batch  = len(sequences)
    padded = torch.full((batch, max_len), pad_value, dtype=torch.long)
    lengths = torch.zeros(batch, dtype=torch.long)
    for i, seq in enumerate(sequences):
        seq = seq[:max_len]
        l   = len(seq)
        if l > 0:
            padded[i, :l] = torch.tensor(seq, dtype=torch.long)
        lengths[i] = max(l, 1)   # avoid zero-length (GRU pack requirement)
    return padded, lengths


def get_episode_data(
    episode: pd.Series,
    pairs_idx: pd.DataFrame,
    max_seq_len: int = MAX_SEQ_LEN,
    device: str = "cpu",
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor,
           torch.Tensor, torch.Tensor, torch.Tensor,
           float]:
    """
    Fetch and tensorise support/query batches for one episode.
    Also returns the episode's SRS value (pre-computed in CELL 10-04).

    The pairs_idx table must have columns:
        prefix_ids (list[int] or stringified list), label_id (int), srs (float)

    Returns:
        sup_seqs, sup_lengths, sup_labels  -- support batch
        qry_seqs, qry_lengths, qry_labels  -- query batch
        episode_srs                        -- float, mean SRS of support pairs
    """
    sup_ids = episode["support_pair_ids"]
    qry_ids = episode["query_pair_ids"]
    episode_srs = float(episode.get("srs", 0.5))

    if isinstance(sup_ids, str):
        sup_ids = ast.literal_eval(sup_ids)
        qry_ids = ast.literal_eval(qry_ids)

    def fetch_batch(pair_ids):
        rows   = pairs_idx.loc[pair_ids]
        seqs   = []
        for prefix in rows["prefix"]:
            if isinstance(prefix, str):
                prefix = ast.literal_eval(prefix)
            seqs.append(list(prefix))
        labels = rows["label"].tolist()
        return seqs, labels

    sup_seqs_list, sup_labels_list = fetch_batch(sup_ids)
    qry_seqs_list, qry_labels_list = fetch_batch(qry_ids)

    sup_seqs, sup_lengths = pad_sequences(sup_seqs_list, max_seq_len)
    qry_seqs, qry_lengths = pad_sequences(qry_seqs_list, max_seq_len)

    sup_labels = torch.tensor(sup_labels_list, dtype=torch.long)
    qry_labels = torch.tensor(qry_labels_list, dtype=torch.long)

    return (
        sup_seqs.to(device),  sup_lengths.to(device),  sup_labels.to(device),
        qry_seqs.to(device),  qry_lengths.to(device),  qry_labels.to(device),
        episode_srs,
    )


cell_end("CELL 10-09", t0)


[CELL 10-09] Define pad_sequences + get_episode_data
[CELL 10-09] start=2026-04-09T15:57:11
[CELL 10-09] elapsed=0.00s  done


In [11]:
# [CELL 10-10] Initialise meta-model (random init -- NO warm-start)
#
# This notebook is the SRS-Adaptive ABLATION: tests C2 alone.
# No pretrained weights are loaded. Compare NB11 for C2 + C3 (warm-start + SRS-Adaptive).

t0 = cell_start("CELL 10-10", "Initialise meta-model (random init)")

seed_everything(GLOBAL_SEED)   # deterministic init

meta_model = GRURecommender(
    n_items=n_items,
    embedding_dim=GRU_CFG["embedding_dim"],
    hidden_dim=GRU_CFG["hidden_dim"],
    num_layers=GRU_CFG["num_layers"],
    dropout=GRU_CFG["dropout"],
).to(DEVICE)

n_params = sum(p.numel() for p in meta_model.parameters())
print(f"[CELL 10-10] Meta-model parameters: {n_params:,}")
print(f"[CELL 10-10] NOTE: Random initialisation -- NO pretrained weights (ablation).")
print(f"[CELL 10-10] NOTE: outer_lr={OUTER_LR} (same as NB07 -- no warm-start to preserve).")

meta_optimizer = torch.optim.Adam(meta_model.parameters(), lr=OUTER_LR)

cell_end("CELL 10-10", t0, n_params=n_params, warmstart=False)


[CELL 10-10] Initialise meta-model (random init)
[CELL 10-10] start=2026-04-09T15:57:12
[CELL 10-10] Meta-model parameters: 367,341
[CELL 10-10] NOTE: Random initialisation -- NO pretrained weights (ablation).
[CELL 10-10] NOTE: outer_lr=0.001 (same as NB07 -- no warm-start to preserve).
[CELL 10-10] n_params=367341
[CELL 10-10] warmstart=False
[CELL 10-10] elapsed=10.51s  done


In [12]:
# [CELL 10-11] SRS-Adaptive MAML training loop
#
# Differences from NB07 (Standard MAML):
#   1. Inner loop: SRS-Adaptive (alpha_i = ALPHA_BASE * srs_i, K_i = f(srs_i))
#   2. Outer loss: SRS-weighted (total_loss += srs_i * query_loss)
#   3. meta_model: random init (no pretrained weights)
#   4. OUTER_LR = 0.001 (same as NB07)
#
# Validation every VAL_EVERY iterations; best HR@10 checkpoint saved.

t0_train = cell_start(
    "CELL 10-11", "SRS-Adaptive FOMAML training loop",
    n_iterations=N_ITER, batch_size=BATCH_SIZE,
    alpha_base=ALPHA_BASE, tau=TAU, k_min=K_MIN, k_max=K_MAX,
    outer_lr=OUTER_LR, warmstart=False, srs_adaptive=True
)

# -- Prepare training index -----------------------------------
train_indices = list(range(len(episodes_train)))

# -- Tracking -------------------------------------------------
train_losses  = []
val_hr10_hist = []
best_val_hr10 = -1.0
best_iter     = 0
best_state    = None

# -- Outer loop -----------------------------------------------
for iteration in range(1, N_ITER + 1):
    meta_model.train()

    # Sample batch of task indices
    batch_idx      = np.random.choice(train_indices, size=min(BATCH_SIZE, len(train_indices)), replace=False)
    batch_episodes = episodes_train.iloc[batch_idx]

    # Build task dicts for outer_step
    batch_tasks = []
    for _, episode in batch_episodes.iterrows():
        try:
            sup_seqs, sup_lengths, sup_labels, \
            qry_seqs, qry_lengths, qry_labels, episode_srs = get_episode_data(
                episode, pairs_srs_idx, MAX_SEQ_LEN, DEVICE
            )
        except (KeyError, ValueError):
            continue

        batch_tasks.append({
            "srs":             episode_srs,
            "support_seqs":    sup_seqs,
            "support_lengths": sup_lengths,
            "support_labels":  sup_labels,
            "query_seqs":      qry_seqs,
            "query_lengths":   qry_lengths,
            "query_labels":    qry_labels,
        })

    if len(batch_tasks) == 0:
        continue

    # SRS-Adaptive outer step
    loss_val = outer_step(meta_model, batch_tasks, meta_optimizer)
    train_losses.append(loss_val)

    # -- Validation every VAL_EVERY iterations ----------------
    if iteration % VAL_EVERY == 0:
        meta_model.train()  # keep train mode — inner loop needs cuDNN backward
        val_scores_all = []
        val_labels_all = []

        for _, ep in episodes_val.iterrows():
            try:
                sup_s, sup_l, sup_lb, \
                qry_s, qry_l, qry_lb, ep_srs = get_episode_data(
                    ep, pairs_srs_idx, MAX_SEQ_LEN, DEVICE
                )
            except (KeyError, ValueError):
                continue

            # SRS-Adaptive inner loop on support
            alpha_v, K_v = get_task_hyperparams(ep_srs)
            params_v = {n: p.clone() for n, p in meta_model.named_parameters()}
            for _step in range(K_v):
                lg_v = functional_forward(meta_model, params_v, sup_s, sup_l)
                ls_v = F.cross_entropy(lg_v, sup_lb)
                gv   = torch.autograd.grad(ls_v, list(params_v.values()), create_graph=False)
                params_v = {n: p - alpha_v * g for (n, p), g in zip(params_v.items(), gv)}

            with torch.no_grad():

                qry_lg = functional_forward(meta_model, params_v, qry_s, qry_l)
            val_scores_all.append(qry_lg.detach().cpu().numpy())
            val_labels_all.extend(qry_lb.cpu().tolist())

        if len(val_labels_all) > 0:
            val_scores_np = np.vstack(val_scores_all)
            val_labels_np = np.array(val_labels_all)
            val_m = compute_metrics(val_scores_np, val_labels_np)
            val_hr10_hist.append((iteration, val_m["hr10"]))

            if val_m["hr10"] > best_val_hr10:
                best_val_hr10 = val_m["hr10"]
                best_iter     = iteration
                best_state    = copy.deepcopy(meta_model.state_dict())
                torch.save(best_state, CHECKPOINT_PATH)

            print(
                f"[CELL 10-11] iter={iteration:4d} "
                f"train_loss={train_losses[-1]:.4f}  "
                f"val_HR@10={val_m['hr10']*100:.2f}%  "
                f"best_HR@10={best_val_hr10*100:.2f}%@{best_iter}"
            )

print(f"\n[CELL 10-11] Training complete.")
print(f"[CELL 10-11] Best val HR@10: {best_val_hr10*100:.2f}% at iteration {best_iter}")
print(f"[CELL 10-11] Checkpoint saved: {CHECKPOINT_PATH}")

cell_end("CELL 10-11", t0_train, best_iter=best_iter, best_val_hr10=f"{best_val_hr10*100:.2f}%")



[CELL 10-11] SRS-Adaptive FOMAML training loop
[CELL 10-11] start=2026-04-09T15:57:22
[CELL 10-11] n_iterations=3000
[CELL 10-11] batch_size=32
[CELL 10-11] alpha_base=0.01
[CELL 10-11] tau=0.5
[CELL 10-11] k_min=3
[CELL 10-11] k_max=7
[CELL 10-11] outer_lr=0.001
[CELL 10-11] warmstart=False
[CELL 10-11] srs_adaptive=True
[CELL 10-11] iter= 100 train_loss=113.4690  val_HR@10=20.42%  best_HR@10=20.42%@100
[CELL 10-11] iter= 200 train_loss=104.0707  val_HR@10=23.55%  best_HR@10=23.55%@200
[CELL 10-11] iter= 300 train_loss=108.1562  val_HR@10=27.32%  best_HR@10=27.32%@300
[CELL 10-11] iter= 400 train_loss=93.9625  val_HR@10=30.71%  best_HR@10=30.71%@400
[CELL 10-11] iter= 500 train_loss=97.3440  val_HR@10=33.38%  best_HR@10=33.38%@500
[CELL 10-11] iter= 600 train_loss=95.0578  val_HR@10=35.44%  best_HR@10=35.44%@600
[CELL 10-11] iter= 700 train_loss=80.6067  val_HR@10=37.22%  best_HR@10=37.22%@700
[CELL 10-11] iter= 800 train_loss=91.0710  val_HR@10=38.42%  best_HR@10=38.42%@800
[CELL 10

In [13]:
# [CELL 10-12] Test evaluation -- load best checkpoint, evaluate on test episodes

t0 = cell_start("CELL 10-12", "Test evaluation (best checkpoint)")

# Load best checkpoint
if not CHECKPOINT_PATH.exists():
    raise RuntimeError(f"Checkpoint missing: {CHECKPOINT_PATH}. Run training cell first.")

meta_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
meta_model.train()  # keep train mode — inner loop needs cuDNN backward
print(f"[CELL 10-12] Loaded best checkpoint from iter {best_iter}: {CHECKPOINT_PATH}")

test_scores_all = []
test_labels_all = []
n_skipped       = 0

for _, ep in episodes_test.iterrows():
    try:
        sup_s, sup_l, sup_lb, \
        qry_s, qry_l, qry_lb, ep_srs = get_episode_data(
            ep, pairs_srs_idx, MAX_SEQ_LEN, DEVICE
        )
    except (KeyError, ValueError):
        n_skipped += 1
        continue

    # SRS-Adaptive inner loop on support set
    alpha_t, K_t = get_task_hyperparams(ep_srs)
    params_t = {n: p.clone() for n, p in meta_model.named_parameters()}
    for _step in range(K_t):
        with torch.enable_grad():
            lg_t = functional_forward(meta_model, params_t, sup_s, sup_l)
            ls_t = F.cross_entropy(lg_t, sup_lb)
            gt   = torch.autograd.grad(ls_t, list(params_t.values()), create_graph=False)
        params_t = {n: p - alpha_t * g for (n, p), g in zip(params_t.items(), gt)}

    with torch.no_grad():
        qry_lg = functional_forward(meta_model, params_t, qry_s, qry_l)
    test_scores_all.append(qry_lg.detach().cpu().numpy())
    test_labels_all.extend(qry_lb.cpu().tolist())

print(f"[CELL 10-12] episodes evaluated: {len(test_scores_all)}, skipped: {n_skipped}")

test_scores_np = np.vstack(test_scores_all)
test_labels_np = np.array(test_labels_all)

test_metrics = compute_metrics(test_scores_np, test_labels_np)

test_hr1    = test_metrics["hr1"]
test_hr5    = test_metrics["hr5"]
test_hr10   = test_metrics["hr10"]
test_ndcg10 = test_metrics["ndcg10"]
test_mrr    = test_metrics["mrr"]

print(f"\n[CELL 10-12] TEST RESULTS (K={K_SUPPORT} support, Q={Q_QUERY} query):")
print(f"  HR@1    = {test_hr1*100:.2f}%")
print(f"  HR@5    = {test_hr5*100:.2f}%")
print(f"  HR@10   = {test_hr10*100:.2f}%   <- PRIMARY")
print(f"  NDCG@10 = {test_ndcg10*100:.2f}%   <- PRIMARY")
print(f"  MRR     = {test_mrr*100:.2f}%")

cell_end("CELL 10-12", t0, n_test_samples=len(test_labels_all))


[CELL 10-12] Test evaluation (best checkpoint)
[CELL 10-12] start=2026-04-09T16:36:24
[CELL 10-12] Loaded best checkpoint from iter 2900: /home/user/jamalla/anonymous-users-mooc-session-meta/models/maml/maml_srs_adaptive_xuetangx.pth
[CELL 10-12] episodes evaluated: 986, skipped: 0

[CELL 10-12] TEST RESULTS (K=5 support, Q=10 query):
  HR@1    = 21.91%
  HR@5    = 37.84%
  HR@10   = 46.20%   <- PRIMARY
  NDCG@10 = 32.96%   <- PRIMARY
  MRR     = 30.07%
[CELL 10-12] n_test_samples=9860
[CELL 10-12] elapsed=18.58s  done


In [14]:
# [CELL 10-13] Standard result block (CLAUDE.md mandatory format)

print("=" * 55)
print(f"RESULTS \u2014 {NOTEBOOK_NAME} \u2014 {DATASET}")
print("=" * 55)
print(f"Protocol:      K={K_SUPPORT} support, Q={Q_QUERY} query")
print(f"Test episodes: {len(episodes_test)}")
print(f"Seed:          {GLOBAL_SEED}")
print()
print(f"HR@1:          {test_hr1*100:.2f}%")
print(f"HR@5:          {test_hr5*100:.2f}%")
print(f"HR@10:         {test_hr10*100:.2f}%   \u2190 PRIMARY")
print(f"NDCG@10:       {test_ndcg10*100:.2f}%   \u2190 PRIMARY")
print(f"MRR:           {test_mrr*100:.2f}%")
print()
print(f"Best val iter: {best_iter}")
print(f"Best val HR@10:{best_val_hr10*100:.2f}%")
print("=" * 55)
print()
print("CONFIG DIFFERENCES FROM NB07 (Standard MAML):")
print(f"  srs_adaptive  = True  (alpha_i=ALPHA_BASE*SRS_i, K_i=f(SRS_i,tau))")
print(f"  outer_loss    = SRS-weighted (sum_i SRS_i * L_query(phi_i))")
print(f"  outer_lr      = {OUTER_LR}  (same as NB07)")
print(f"  warm_start    = False (random init -- ablation only)")
print(f"  tau           = {TAU}  K_min={K_MIN} (high-SRS)  K_max={K_MAX} (low-SRS)")

RESULTS — 10_srs_adaptive_maml_xuetangx — xuetangx
Protocol:      K=5 support, Q=10 query
Test episodes: 986
Seed:          20260107

HR@1:          21.91%
HR@5:          37.84%
HR@10:         46.20%   ← PRIMARY
NDCG@10:       32.96%   ← PRIMARY
MRR:           30.07%

Best val iter: 2900
Best val HR@10:46.91%

CONFIG DIFFERENCES FROM NB07 (Standard MAML):
  srs_adaptive  = True  (alpha_i=ALPHA_BASE*SRS_i, K_i=f(SRS_i,tau))
  outer_loss    = SRS-weighted (sum_i SRS_i * L_query(phi_i))
  outer_lr      = 0.001  (same as NB07)
  warm_start    = False (random init -- ablation only)
  tau           = 0.5  K_min=3 (high-SRS)  K_max=7 (low-SRS)


In [15]:
# [CELL 10-14] Save results JSON

t0 = cell_start("CELL 10-14", "Save results JSON")

results = {
    "run_id":     RUN_ID,
    "notebook":   NOTEBOOK_NAME,
    "dataset":    DATASET,
    "run_tag":    RUN_TAG,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "config": {
        "K_support":    K_SUPPORT,
        "Q_query":      Q_QUERY,
        "alpha_base":   ALPHA_BASE,
        "tau":          TAU,
        "k_min":        K_MIN,
        "k_max":        K_MAX,
        "outer_lr":     OUTER_LR,
        "batch_size":   BATCH_SIZE,
        "n_iterations": N_ITER,
        "global_seed":  GLOBAL_SEED,
        "gru_cfg":      GRU_CFG,
        "warmstart":    False,
        "srs_adaptive": True,
    },
    "test_metrics": {
        "hr1":    test_hr1,
        "hr5":    test_hr5,
        "hr10":   test_hr10,
        "ndcg10": test_ndcg10,
        "mrr":    test_mrr,
        "n":      int(len(test_labels_all)),
    },
    "val_best": {
        "best_iter":     best_iter,
        "best_val_hr10": best_val_hr10,
    },
    "checkpoint": str(CHECKPOINT_PATH),
}

write_json_atomic(RESULTS_PATH, results)
print(f"[CELL 10-14] Results saved: {RESULTS_PATH}")

# Update report.json
report = read_json(REPORT_PATH)
report["metrics"] = results["test_metrics"]
report["key_findings"] = [
    f"SRS-Adaptive MAML ablation: HR@10={test_hr10*100:.2f}%, NDCG@10={test_ndcg10*100:.2f}%",
    f"Best val HR@10={best_val_hr10*100:.2f}% at iteration {best_iter}",
    f"Hyperparams: alpha_base={ALPHA_BASE}, tau={TAU}, K_min={K_MIN}, K_max={K_MAX}, outer_lr={OUTER_LR}, batch={BATCH_SIZE}, iters={N_ITER}",
    f"No warm-start (random init) -- ablation of C2 alone",
    f"Inner loop: SRS-Adaptive (alpha_i=alpha_base*SRS_i, K_i depends on tau)",
    f"Outer loss: SRS-weighted (sum_i SRS_i * L_query(phi_i))",
    f"Test episodes evaluated: {len(test_labels_all)}, skipped: {n_skipped}",
]
write_json_atomic(REPORT_PATH, report)

# Update manifest.json
manifest = read_json(MANIFEST_PATH)
manifest["artifacts"].extend([
    {"type": "results_json", "path": str(RESULTS_PATH)},
    {"type": "checkpoint",   "path": str(CHECKPOINT_PATH)},
    {"type": "report",       "path": str(REPORT_PATH)},
    {"type": "config",       "path": str(CONFIG_PATH)},
])
write_json_atomic(MANIFEST_PATH, manifest)

cell_end("CELL 10-14", t0)


[CELL 10-14] Save results JSON
[CELL 10-14] start=2026-04-09T16:36:43
[CELL 10-14] Results saved: /home/user/jamalla/anonymous-users-mooc-session-meta/reports/10_srs_adaptive_maml_xuetangx/20260409_155705/results_10_srs_adaptive_maml_xuetangx.json
[CELL 10-14] elapsed=0.02s  done


In [16]:
# [CELL 10-15] Update PROJECT_STATE.md with results

t0 = cell_start("CELL 10-15", "Update PROJECT_STATE.md")

state_path = PATHS["PROJECT_STATE"]

result_block = (
    f"\n\n## {NOTEBOOK_NAME} \u2014 Results\n"
    f"Run: {RUN_TAG}  |  Seed: {GLOBAL_SEED}\n"
    f"Protocol: K={K_SUPPORT} support, Q={Q_QUERY} query | Test episodes: {len(episodes_test)}\n"
    f"Ablation: SRS-Adaptive MAML (no warm-start) | alpha_base={ALPHA_BASE}, tau={TAU}, K_min={K_MIN}, K_max={K_MAX}\n"
    f"\n"
    f"| Metric   | Value    |\n"
    f"|----------|----------|\n"
    f"| HR@1     | {test_hr1*100:.2f}%  |\n"
    f"| HR@5     | {test_hr5*100:.2f}%  |\n"
    f"| HR@10    | {test_hr10*100:.2f}%  |\n"
    f"| NDCG@10  | {test_ndcg10*100:.2f}%  |\n"
    f"| MRR      | {test_mrr*100:.2f}%  |\n"
    f"\nBest val HR@10: {best_val_hr10*100:.2f}% @ iter {best_iter}\n"
    f"Checkpoint: {CHECKPOINT_PATH}\n"
)

if state_path.exists():
    existing = state_path.read_text(encoding="utf-8")
    marker = f"## {NOTEBOOK_NAME}"
    if marker in existing:
        # Replace existing block
        lines = existing.split("\n")
        new_lines = []
        skip = False
        for line in lines:
            if line.startswith(marker):
                skip = True
            elif skip and line.startswith("## ") and not line.startswith(marker):
                skip = False
            if not skip:
                new_lines.append(line)
        updated = "\n".join(new_lines) + result_block
    else:
        updated = existing + result_block
    state_path.write_text(updated, encoding="utf-8")
    print(f"[CELL 10-15] PROJECT_STATE.md updated: {state_path}")
else:
    state_path.write_text(result_block, encoding="utf-8")
    print(f"[CELL 10-15] PROJECT_STATE.md created: {state_path}")

cell_end("CELL 10-15", t0)


[CELL 10-15] Update PROJECT_STATE.md
[CELL 10-15] start=2026-04-09T16:36:43
[CELL 10-15] PROJECT_STATE.md updated: /home/user/jamalla/anonymous-users-mooc-session-meta/PROJECT_STATE.md
[CELL 10-15] elapsed=0.02s  done


## Notebook 10 Complete — SRS-Adaptive MAML Ablation (XuetangX)

**What was done:**
- Attached mean SRS scores to episodes (mean test episode SRS = 0.5355)
- Initialised GRURecommender from **random weights** (no pretrained backbone)
- Trained FOMAML with SRS-Adaptive inner loop for 3000 iterations
- Best HR@10 checkpoint saved at iter **2900**

**SRS-Adaptive inner loop (C2):**
```
alpha_i = ALPHA_BASE * SRS_i     # larger alpha for reliable sessions
K_i     = K_MIN if SRS_i >= TAU else K_MAX
ALPHA_BASE=0.01  TAU=0.5  K_MIN=3  K_MAX=7
```

**Test Results (K=5 support, Q=10 query, 986 episodes):**
| Metric | Value |
|--------|-------|
| HR@1 | 21.91% |
| HR@5 | 37.84% |
| **HR@10** | **46.20%** |
| **NDCG@10** | **32.96%** |
| MRR | 30.07% |

→ SRS-Adaptive alone (no warm-start) is slightly below Standard MAML (NB07: 47.46%).
Full benefit requires combining with warm-start (NB11).

**Next:** `11_warmstart_srs_adaptive_maml_xuetangx.ipynb` — MAIN CONTRIBUTION (C3)
